# Project 1 - Medical Text Summarisation

## Data loading

In [18]:
## Import libraries
import os
import tarfile
from pathlib import Path
import pandas as pd

In [19]:
# Defining file paths

project_root = Path.cwd().parent
data_dir = project_root / "data"
archive_path = data_dir / "NLMCXR_reports.tgz"
extract_dir = data_dir / "nlmcxr_reports"

print("Project root:", project_root)
print("Data directory:", data_dir)
print("Archive path:", archive_path)
print("Archive exists:", archive_path.exists())
print("Extract folder:", extract_dir)

Project root: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser
Data directory: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data
Archive path: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\NLMCXR_reports.tgz
Archive exists: True
Extract folder: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports


In [20]:
# Extract the .tgz archive
extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print("Extraction complete.")
print("Files extracted to:", extract_dir)

Extraction complete.
Files extracted to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports


In [21]:
# Inspect the extracted files
all_files = list(extract_dir.rglob("*"))

print("Total extracted items:", len(all_files))

for file in all_files[:20]:
    print(file)

Total extracted items: 3956
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\10.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\100.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1000.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1001.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1002.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1003.xml
c:\Users\THE AGAMBISAS\Desktop\Practical Pr

In [22]:
# Checking the file types that are present
xml_files = list(extract_dir.rglob("*.xml"))
txt_files = list(extract_dir.rglob("*.txt"))

print("Number of XML files:", len(xml_files))
print("Number of TXT files:", len(txt_files))

Number of XML files: 3955
Number of TXT files: 0


In [23]:
# Open to view a sample report
sample_file = xml_files[0]
print("Sample file path:", sample_file)

with open(sample_file, "r", encoding="utf-8") as f:
    content = f.read()

print(content[:2000])

Sample file path: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\nlmcxr_reports\ecgen-radiology\1.xml
<?xml version="1.0" encoding="utf-8"?>
<eCitation>
   <meta type="rr"/>
   <uId id="CXR1"/>
   <pmcId id="1"/>
   <docSource>CXR</docSource>
   <IUXRId id="1"/>
   <licenseType>open-access</licenseType>
   <licenseURL>http://creativecommons.org/licenses/by-nc-nd/4.0/</licenseURL>
   <ccLicense>byncnd</ccLicense>
   <articleURL/>
   <articleDate>2013-08-01</articleDate>
   <articleType>XR</articleType>
   <publisher>Indiana University</publisher>
   <title>Indiana University Chest X-ray Collection</title>
   <note>The data are drawn from multiple hospital systems.</note>
   <specialty>pulmonary diseases</specialty>
   <subset>CXR</subset>
   <MedlineCitation Owner="Indiana University" Status="supplied by publisher">
   
      <Article PubModel="Electronic">
      
         <Journal>
         
            <JournalIssue>
            
               <PubDate>

In [24]:
# Looking for whether the words FINDINGS and IMPRESSION appear in the sample file content

print("FINDINGS" in content)
print("IMPRESSION" in content)

True
True


### Summary Data Loading
I imported the required libraries, defined the dataset paths, extracted the compressed report archive, inspected the extracted files and opened a sample report to understand its structure. This helped confirmed whether the report files contain the 'FINDINGS' and 'IMPRESSION' fields needed for the summarisation task.

## Process Raw Reports into a flat file for modelling

In [25]:
# Import Libraries
import xml.etree.ElementTree as ET

# Define paths
output_file = data_dir / "processed_reports.csv"

print("Output fill will be:", output_file)

Output fill will be: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\processed_reports.csv


In [26]:
# Function to extract FINDINGS and IMPRESSION from the file
def extract_report_sections(xml_file):
    findings_text = None
    impression_text = None

    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        for abstract_text in root.iter("AbstractText"):
            label = abstract_text.attrib.get("Label", "").strip().upper()
            text = "".join(abstract_text.itertext()).strip()

            if label == "FINDINGS":
                findings_text = text
            elif label == "IMPRESSION":
                impression_text = text
            
        return findings_text, impression_text
    
    except Exception as e:
        return None, None


In [27]:
# Loooping through all report in the file to extract findings and impression

records = []

for xml_file in xml_files:
    findings, impression = extract_report_sections(xml_file)

    records.append({
        "file_name": xml_file.name,
        "findings": findings,
        "impression": impression
    })

print("Total records collected:", len(records))
print("First record:")
print(records[0])

Total records collected: 3955
First record:
{'file_name': '1.xml', 'findings': 'The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.', 'impression': 'Normal chest x-XXXX.'}


In [28]:
df = pd.DataFrame(records) # dataframe

print("DataFrame shape:", df.shape)
print(df.isna().sum())
df.head()

DataFrame shape: (3955, 3)
file_name     0
findings      0
impression    0
dtype: int64


,file_name,findings,impression
0,1.xml,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,10.xml,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,100.xml,Both lungs are clear and expanded. Heart and m...,No active disease.
3,1000.xml,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,1001.xml,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


In [29]:
# Cleaning the data
df = df.dropna(subset=["findings", "impression"])
df["findings"] = df["findings"].str.strip()
df["impression"] = df["impression"].str.strip()

df = df[(df["findings"] != "") & (df["impression"] != "")]

print("Shape after cleaning:", df.shape)
df.head()

Shape after cleaning: (3419, 3)


,file_name,findings,impression
0,1.xml,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,10.xml,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,100.xml,Both lungs are clear and expanded. Heart and m...,No active disease.
3,1000.xml,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,1001.xml,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


In [30]:
# Save the file
df.to_csv(output_file, index=False)

print("Flat file saved successfully.")
print("Saved to:", output_file)

Flat file saved successfully.
Saved to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\processed_reports.csv


In [31]:
# Inspect the results

print(df.info())
df.sample(5, random_state=42)

<class 'pandas.DataFrame'>
Index: 3419 entries, 0 to 3954
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   file_name   3419 non-null   str  
 1   findings    3419 non-null   str  
 2   impression  3419 non-null   str  
dtypes: str(3)
memory usage: 106.8 KB
None


,file_name,findings,impression
2277,3076.xml,"Frontal view kyphotic and rotated, low lung vo...","Low lung volumes, otherwise, no definite acute..."
3506,591.xml,The cardiomediastinal silhouette is within nor...,1. No acute cardiopulmonary process. 2. Age-in...
196,1177.xml,Normal heart. Clear lungs. Trachea midline. Sc...,No acute cardiopulmonary abnormality.
3821,879.xml,Cardiac and mediastinal contours are within no...,Negative chest x-XXXX.
1342,2222.xml,"Lungs are clear bilaterally, with no focal con...",1. Round opacity measuring 2 cm in diameter wi...


In [32]:
# Keeping only the findings and impression columns for modelling

model_df = df[["findings","impression"]].copy()
model_output_file = data_dir / "modeling_data.csv"

model_df.to_csv(model_output_file, index=False)

print("Modeling flat file saved to:", model_output_file)
model_df.head()

Modeling flat file saved to: c:\Users\THE AGAMBISAS\Desktop\Practical Project\medical-text-summariser\data\modeling_data.csv


,findings,impression
0,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.
2,Both lungs are clear and expanded. Heart and m...,No active disease.
3,There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...
4,Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.


### Summary: Processing raw data

I processed the raw XML reports into a flat file for modeling. I searched through all the report files, parsed each XML document, extracted the FINDINGS and IMPRESSION sections, stored them in a pandas DataFrame, removed incomplete records and saved the cleaned dataset as a CSV file